In [2]:
import os, math, time, copy, random, sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models

# Optional for report/CM (not required for training)
from sklearn.metrics import classification_report, confusion_matrix
import itertools

In [26]:
# ------------------------
# CONFIG
# ------------------------
# Try to auto-detect the dataset folder inside /kaggle/input
INPUT_ROOT = None
for p in Path("/kaggle/input").iterdir():
    if p.is_dir() and (p / "Training").exists() and (p / "Validation").exists():
        INPUT_ROOT = str(p)
        break

assert INPUT_ROOT is not None, "Could not find dataset with Training/ and Validation/ under /kaggle/input!"

TRAIN_DIR   = os.path.join(INPUT_ROOT, "Training")
VAL_DIR     = os.path.join(INPUT_ROOT, "Validation")
WORK_DIR    = "/kaggle/working"

IMG_SIZE    = 224
BATCH_SIZE  = 128
EPOCHS      = 10
BASE_LR     = 3e-4
FREEZE_BACKBONE_EPOCHS = 2
WEIGHTED_SAMPLER       = True   # helps when male/female counts differ
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print("Dataset root:", INPUT_ROOT)

Using device: cuda
Dataset root: /kaggle/input/gender-classification-dataset


In [27]:
# ------------------------
# REPRODUCIBILITY
# ------------------------
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
set_seed()

In [29]:
# ------------------------
# TRANSFORMS
# ------------------------
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.1)], p=0.5),
    transforms.RandomRotation(12),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


In [38]:
# ------------------------
# DATASETS & LOADERS
# ------------------------
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=val_tfms)

class_names = train_ds.classes
num_classes = len(class_names)
print("Classes:", class_names)
print(f"Train images: {len(train_ds)}, Val images: {len(val_ds)}")

Classes: ['female', 'male']
Train images: 47009, Val images: 11649


In [30]:
if WEIGHTED_SAMPLER:
    targets = [y for _, y in train_ds.samples]
    binc = torch.bincount(torch.tensor(targets))
    class_weight = 1.0 / binc.float()
    sample_weight = [class_weight[t] for t in targets]
    sampler = WeightedRandomSampler(weights=sample_weight,
                                    num_samples=len(sample_weight),
                                    replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

In [31]:
# ------------------------
# MODEL (Robust Transfer Learning)
# ------------------------
def build_model(num_classes):
    # try to load pretrained weights; if not possible (no internet, not cached), fall back to random init
    try:
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        print("Loaded ResNet18 with ImageNet pretrained weights.")
    except Exception as e:
        print("WARNING: Could not load pretrained weights (likely no internet). "
              "Proceeding with randomly initialized ResNet18. "
              "For best accuracy, enable Internet and re-run once.")
        model = models.resnet18(weights=None)
    in_f = model.fc.in_features
    model.fc = nn.Linear(in_f, num_classes)
    return model

model = build_model(num_classes).to(DEVICE)

Loaded ResNet18 with ImageNet pretrained weights.


In [32]:
# ------------------------
# OPTIMIZER / SCHEDULER / LOSS
# ------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-4)

steps_per_epoch = math.ceil(len(train_loader.dataset) / BATCH_SIZE)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=BASE_LR, epochs=EPOCHS, steps_per_epoch=max(1, steps_per_epoch)
)

In [33]:
# ------------------------
# TRAIN / EVAL
# ------------------------
def run_epoch(loader, train=True):
    model.train(mode=train)
    running_loss, correct, total = 0.0, 0, 0
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train: optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out = model(x)
            loss = criterion(out, y)
        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        running_loss += loss.item() * x.size(0)
        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return running_loss/total, correct/total


In [34]:
best_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0
no_improve = 0
PATIENCE = 5

In [35]:
t0 = time.time()
for epoch in range(1, EPOCHS+1):
    # warmup with a frozen backbone (helps when using pretrained)
    if epoch <= FREEZE_BACKBONE_EPOCHS:
        for name, p in model.named_parameters():
            if not name.startswith("fc"): p.requires_grad = False
    else:
        for p in model.parameters(): p.requires_grad = True

    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train L {train_loss:.4f} A {train_acc:.4f} | "
          f"Val L {val_loss:.4f} A {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(model.state_dict())
        torch.save(best_wts, os.path.join(WORK_DIR, "gender_resnet18_best.pth"))
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

/tmp/ipykernel_36/4197889234.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_36/4197889234.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 01/10 | Train L 0.6232 A 0.6623 | Val L 0.5230 A 0.7771
Epoch 02/10 | Train L 0.4689 A 0.8004 | Val L 0.4134 A 0.8055
Epoch 03/10 | Train L 0.1151 A 0.9593 | Val L 0.0785 A 0.9718
Epoch 04/10 | Train L 0.0847 A 0.9706 | Val L 0.0823 A 0.9711
Epoch 05/10 | Train L 0.0761 A 0.9730 | Val L 0.0669 A 0.9752
Epoch 06/10 | Train L 0.0650 A 0.9767 | Val L 0.0657 A 0.9741
Epoch 07/10 | Train L 0.0581 A 0.9795 | Val L 0.0638 A 0.9761
Epoch 08/10 | Train L 0.0468 A 0.9826 | Val L 0.0648 A 0.9765
Epoch 09/10 | Train L 0.0390 A 0.9859 | Val L 0.0642 A 0.9775
Epoch 10/10 | Train L 0.0316 A 0.9884 | Val L 0.0673 A 0.9770


In [36]:
print(f"Training finished in {(time.time()-t0)/60:.1f} min | Best Val Acc: {best_acc:.4f}")
model.load_state_dict(best_wts)
print("Best model saved to /kaggle/working/gender_resnet18_best.pth")

Training finished in 37.3 min | Best Val Acc: 0.9775
Best model saved to /kaggle/working/gender_resnet18_best.pth
